# 02 Extraction PDF, sections et tableaux

**Objectif** : examiner les PDF placés dans `data/raw/`, comparer les parseurs et vérifier la tracabilité jusqu'à  la cellule.

**Critère de passage** : le parseur choisit une sortie exploitable ; les tableaux critiques sont revus avant indexation.

In [1]:
from pathlib import Path
import sys

root_hint = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root_hint / 'notebooks'))
from helpers import bootstrap, display_table, module_available

ROOT = bootstrap()
RAW_DIR = ROOT / 'data' / 'raw'
pdf_files = sorted(RAW_DIR.glob('*.pdf'))
group_qrt_pdf = RAW_DIR / 'foyer-groupe-qrt-public-2025.pdf'
{
    'raw_directory': str(RAW_DIR),
    'pdf_count': len(pdf_files),
    'pymupdf': module_available('pymupdf') or module_available('fitz'),
    'pdfplumber': module_available('pdfplumber'),
    'camelot': module_available('camelot'),
    'docling': module_available('docling'),
}

{'raw_directory': 'C:\\Users\\choun\\Downloads\\Prudential_Evidence_Lab_MVP_Source\\prudential_evidence_lab\\data\\raw',
 'pdf_count': 11,
 'pymupdf': True,
 'pdfplumber': False,
 'camelot': False,
 'docling': True}

In [2]:
page_rows = []
if pdf_files and (module_available('pymupdf') or module_available('fitz')):
    try:
        import pymupdf as fitz
    except ImportError:
        import fitz
    with fitz.open(group_qrt_pdf) as document:
        for page_index in range(len(document)):
            page = document[page_index]
            text = page.get_text('text')
            page_rows.append({
                'page': page_index + 1,
                'characters': len(text),
                'blocks': len(page.get_text('blocks')),
                'potential_scan': len(text.strip()) < 40,
            })
else:
    print('Placer un PDF public dans data/raw/ et installer .[ingestion].')
display_table(page_rows)

,page,characters,blocks,potential_scan
0,1,2101,46,False
1,2,1929,44,False
2,3,2624,49,False
3,4,1383,39,False
4,5,706,23,False
5,6,4013,51,False
6,7,2271,23,False
7,8,823,16,False
8,9,3098,47,False
9,10,2577,45,False


In [3]:
table_rows = []
if pdf_files and module_available('pdfplumber'):
    import pdfplumber
    with pdfplumber.open(group_qrt_pdf) as document:
        for page_index, page in enumerate(document.pages, start=1):
            for table_index, table in enumerate(page.extract_tables() or [], start=1):
                table_rows.append({
                    'page': page_index,
                    'table': table_index,
                    'rows': len(table),
                    'columns': max((len(row) for row in table), default=0),
                    'preview': str(table[:2])[:180],
                })
display_table(table_rows)

# pdfplumber est ici un diagnostic exploratoire : le nombre de lignes qu'il reconstruit varie selon sa version. Le contrat est vérifié sur les cellules.

from ingestion.pymupdf_fallback import extract_qrt_coverage_table

if pdf_files:
    qrt_table, qrt_warnings = extract_qrt_coverage_table(
        group_qrt_pdf, entity='Groupe Foyer', period='2025'
    )
    assert qrt_table is not None, f'Extraction QRT impossible : {qrt_warnings}'
    extracted = {cell.row_code: cell for cell in qrt_table.cells}
    assert set(extracted) == {'R0660', 'R0680', 'R0690'}
    assert all(cell.column_code == 'C0010' for cell in extracted.values())
    assert extracted['R0660'].normalized_value == 2407647.0
    assert extracted['R0680'].normalized_value == 840040.0
    assert extracted['R0690'].normalized_value == 2.87
    display_table([cell.model_dump() for cell in qrt_table.cells])

In [4]:
from app.store.artifacts import store

localized_cells = [
    {
        'document': chunk.document_id,
        'page': chunk.locator.page,
        'table': chunk.locator.table_id,
        'row': chunk.locator.row,
        'column': chunk.locator.column,
        'bbox': chunk.locator.bbox,
    }
    for chunk in store.chunks if chunk.locator.table_id
]
qrt_cells = [cell for cell in localized_cells if cell['document'] == 'foyer_group_qrt_2025']
assert {cell['row'] for cell in qrt_cells} == {'R0660', 'R0680', 'R0690'}
assert all(cell['column'] == 'C0010' and cell['page'] == 7 for cell in qrt_cells)
display_table(localized_cells)

,document,page,table,row,column,bbox
0,table_extraction_demo,1,demo-table-1,2,3,"[120.0, 240.0, 260.0, 278.0]"
1,foyer_group_qrt_2025,7,S.23.01.22,R0660,C0010,"[495.6, 270.13, 564.39, 276.85]"
2,foyer_group_qrt_2025,7,S.23.01.22,R0680,C0010,"[495.6, 278.41, 564.37, 285.13]"
3,foyer_group_qrt_2025,7,S.23.01.22,R0690,C0010,"[495.6, 286.93, 564.33, 293.65]"
4,foyer_assurances_qrt_2025,11,S.23.01.01,R0540,C0010,"[439.92, 490.85, 513.22, 498.75]"
5,foyer_assurances_qrt_2025,11,S.23.01.01,R0580,C0010,"[439.92, 510.53, 513.22, 518.43]"
6,foyer_assurances_qrt_2025,11,S.23.01.01,R0620,C0010,"[439.92, 530.21, 513.23, 538.11]"
7,foyer_global_health_qrt_2025,8,S.23.01.01,R0540,C0010,"[429.72, 480.05, 501.26, 487.8]"
8,foyer_global_health_qrt_2025,8,S.23.01.01,R0580,C0010,"[429.72, 499.25, 501.26, 507.0]"
9,foyer_global_health_qrt_2025,8,S.23.01.01,R0620,C0010,"[429.72, 518.45, 501.3, 526.2]"


## All cached table evidence contracts

This view compares group and solo-entity QRT templates and prints every verified cell. It does not claim that every PDF table has been converted into typed evidence.

In [5]:
from ingestion.exporters import read_result

processed_root = ROOT / 'data' / 'processed'
cached_results = [
    read_result(directory)
    for directory in sorted(processed_root.iterdir())
    if directory.is_dir() and (directory / 'manifest.json').exists()
]
all_verified_cells = [
    {
        'document_id': cached.manifest.document_id,
        'entity': table.entity,
        'period': table.period,
        'table_id': table.id,
        'row_code': cell.row_code,
        'column_code': cell.column_code,
        'label': cell.row_label,
        'raw_value': cell.raw_value,
        'normalized_value': cell.normalized_value,
        'unit': cell.unit,
        'page': cell.provenance.page,
        'bbox': cell.provenance.bbox,
        'source_kind': cell.provenance.source_kind,
    }
    for cached in cached_results
    for table in cached.tables
    for cell in table.cells
]
group_rows = {item['row_code'] for item in all_verified_cells if item['table_id'] == 'S.23.01.22'}
solo_rows = {item['row_code'] for item in all_verified_cells if item['table_id'] == 'S.23.01.01'}
assert group_rows == {'R0660', 'R0680', 'R0690'}
assert not solo_rows or solo_rows == {'R0540', 'R0580', 'R0620'}
display_table(all_verified_cells)

,document_id,entity,period,table_id,row_code,column_code,label,raw_value,normalized_value,unit,page,bbox,source_kind
0,foyer_assurances_qrt_2025,Foyer Assurances S.A.,2025,S.23.01.01,R0540,C0010,Eligible own funds to meet the SCR,477.591,477591.00,thousand EUR,11,"[439.92, 490.85, 513.22, 498.75]",native_pdf_text_verified
1,foyer_assurances_qrt_2025,Foyer Assurances S.A.,2025,S.23.01.01,R0580,C0010,SCR,211.981,211981.00,thousand EUR,11,"[439.92, 510.53, 513.22, 518.43]",native_pdf_text_verified
2,foyer_assurances_qrt_2025,Foyer Assurances S.A.,2025,S.23.01.01,R0620,C0010,Ratio of eligible own funds to SCR,"2,25",2.25,ratio,11,"[439.92, 530.21, 513.23, 538.11]",native_pdf_text_verified
3,foyer_global_health_qrt_2025,Foyer Global Health S.A.,2025,S.23.01.01,R0540,C0010,Eligible own funds to meet the SCR,78.014,78014.00,thousand EUR,8,"[429.72, 480.05, 501.26, 487.8]",native_pdf_text_verified
4,foyer_global_health_qrt_2025,Foyer Global Health S.A.,2025,S.23.01.01,R0580,C0010,SCR,29.168,29168.00,thousand EUR,8,"[429.72, 499.25, 501.26, 507.0]",native_pdf_text_verified
5,foyer_global_health_qrt_2025,Foyer Global Health S.A.,2025,S.23.01.01,R0620,C0010,Ratio of eligible own funds to SCR,"2,67",2.67,ratio,8,"[429.72, 518.45, 501.3, 526.2]",native_pdf_text_verified
6,foyer_group_qrt_2025,Groupe Foyer,2025,S.23.01.22,R0660,C0010,Total des fonds propres éligibles pour couvrir...,2.407.647,2407647.00,milliers EUR,7,"[495.6, 270.13, 564.39, 276.85]",native_pdf_text_verified
7,foyer_group_qrt_2025,Groupe Foyer,2025,S.23.01.22,R0680,C0010,Capital de solvabilité requis total du groupe,840.040,840040.00,milliers EUR,7,"[495.6, 278.41, 564.37, 285.13]",native_pdf_text_verified
8,foyer_group_qrt_2025,Groupe Foyer,2025,S.23.01.22,R0690,C0010,Ratio total des fonds propres éligibles sur SC...,"2,87",2.87,ratio,7,"[495.6, 286.93, 564.33, 293.65]",native_pdf_text_verified
9,foyer_group_qrt_2025_docling_test,Groupe Foyer,2025,S.23.01.22,R0660,C0010,Total des fonds propres éligibles pour couvrir...,2.407.647,2407647.00,milliers EUR,7,"[495.6, 270.13, 564.39, 276.85]",native_pdf_text_verified
